In [1]:
!pip install kaggle

     ---------------------------------------- 76.8/76.8 kB 1.4 MB/s eta 0:00:00
  Using cached bleach-6.3.0-py3-none-any.whl (164 kB)
     -------------------------------------- 201.8/201.8 kB 3.1 MB/s eta 0:00:00
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl (437 kB)
  Using cached requests-2.33.1-py3-none-any.whl (64 kB)
     ---------------------------------------- 78.4/78.4 kB ? eta 0:00:00
  Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  Using cached webencodings-0.5.1-py2.py3-none-any.whl (11 kB)
     ---------------------------------------- 78.2/78.2 kB ? eta 0:00:00
  Using cached charset_normalizer-3.4.7-cp311-cp311-win_amd64.whl (159 kB)
  Using cached idna-3.11-py3-none-any.whl (71 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
print(os.listdir("."))

['aptos2019-blindness-detection.zip', 'my_env', 'retinopathy.ipynb', 'sample_submission.csv', 'test.csv', 'test_images', 'train.csv', 'train_images']


In [3]:
import os

train_path = "train_images"
test_path = "test_images"

In [4]:
train_count = len(os.listdir(train_path))
test_count = len(os.listdir(test_path))

print("Train images:", train_count)
print("Test images:", test_count)

Train images: 3662
Test images: 1928


In [8]:
import os, sys, warnings, zipfile, random, time
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3, ResNet50, DenseNet121
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
)
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    cohen_kappa_score, classification_report,
    confusion_matrix, roc_auc_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

# -----------------------------
# SETTINGS
# -----------------------------
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Fix random seeds (reproducibility)
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# DEVICE CHECK (SAFE VERSION)
# -----------------------------
print("=" * 65)
print("  Diabetic Retinopathy Detection Pipeline — TensorFlow/Keras")
print("=" * 65)

print(f"  TensorFlow : {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"  Device     : GPU x{len(gpus)} ✓")
    except Exception as e:
        print("  GPU detected but configuration failed:", e)
        print("  Falling back to CPU")
else:
    print("  Device     : CPU (training will be slower)")

print("=" * 65)

  Diabetic Retinopathy Detection Pipeline — TensorFlow/Keras
  TensorFlow : 2.21.0
  Device     : CPU (training will be slower)


In [9]:
DATA_ROOT = "."

TRAIN_CSV    = os.path.join(DATA_ROOT, "train.csv")
TEST_CSV     = os.path.join(DATA_ROOT, "test.csv")

TRAIN_IMGS   = os.path.join(DATA_ROOT, "train_images")
TEST_IMGS    = os.path.join(DATA_ROOT, "test_images")

MODELS_DIR   = "./saved_models"
PLOTS_DIR    = "./plots"
HISTORY_CSV  = "./training_history.csv"
SUBMISSION   = "./submission.csv"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

CONFIG = {
    "img_size":    224,
    "batch_size":  16,
    "epochs":      20,
    "lr":          1e-4,
    "weight_decay":1e-5,
    "num_classes": 5,
    "dropout":     0.4,
    "val_split":   0.20,
    "seed":        42,
    "num_workers": 0,
}

tf.random.set_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
random.seed(CONFIG["seed"])

In [10]:
CLASS_NAMES = ["No DR", "Mild NPDR", "Moderate NPDR", "Severe NPDR", "PDR"]
CLASS_SHORT = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
BAR_COLORS  = ["#27500A", "#BA7517", "#EF9F27", "#E24B4A", "#A32D2D"]

# ── XAI keyword map ───────────────────────────────────────────────────
KEYWORD_MAP = {
    0: {
        "base":       ["clear retinal vessels", "no lesions", "healthy optic disc", "normal macula"],
        "macula":     ["normal foveal reflex", "clear central vision zone"],
        "optic_disc": ["healthy disc margins", "normal cup-to-disc ratio"],
        "peripheral": ["no peripheral hemorrhages"],
        "superior":   ["no superior vessel abnormalities"],
        "inferior":   ["no inferior lesions"],
    },
    1: {
        "base":       ["microaneurysms", "dot hemorrhages", "early vessel changes"],
        "macula":     ["peri-foveal microaneurysms", "macular proximity lesions"],
        "optic_disc": ["peripapillary hemorrhages", "disc-margin vessel changes"],
        "peripheral": ["scattered peripheral microaneurysms"],
        "superior":   ["superior quadrant dot hemorrhages"],
        "inferior":   ["inferior retinal microaneurysms"],
    },
    2: {
        "base":       ["microaneurysms", "intraretinal hemorrhages", "hard exudates", "cotton-wool spots"],
        "macula":     ["macular hard exudates", "CSME risk", "foveal threat"],
        "optic_disc": ["peripapillary hard exudates", "nerve fiber layer hemorrhages"],
        "peripheral": ["blot hemorrhages", "venous beading"],
        "superior":   ["superior flame-shaped hemorrhages"],
        "inferior":   ["inferior IRMA"],
    },
    3: {
        "base":       ["extensive hemorrhages", "venous beading", "IRMA", "cotton-wool spots"],
        "macula":     ["macular ischemia", "severe macular edema"],
        "optic_disc": ["disc neovascularization risk", "peripapillary IRMA"],
        "peripheral": ["4-quadrant hemorrhages", "extensive peripheral IRMA"],
        "superior":   ["superior venous beading"],
        "inferior":   ["inferior vessel occlusion signs"],
    },
    4: {
        "base":       ["neovascularization", "fibrous proliferation", "vitreous hemorrhage risk", "retinal traction"],
        "macula":     ["tractional macular detachment risk", "NVD near disc"],
        "optic_disc": ["neovascularization of the disc (NVD)", "disc fibrovascular proliferation"],
        "peripheral": ["neovascularization elsewhere (NVE)", "pre-retinal hemorrhage"],
        "superior":   ["superior fibrovascular membranes"],
        "inferior":   ["inferior retinal traction"],
    },
}

EXPLANATION_TEMPLATES = {
    0: ("The retinal image shows no signs of diabetic retinopathy. "
        "Analysis reveals {kw}. "
        "The Grad-CAM heatmap confirms no abnormal activation in critical retinal zones."),
    1: ("Mild NPDR detected with {conf:.0%} confidence. "
        "The heatmap highlights the {zones}, identifying {kw}. "
        "These are early microvascular changes, not immediately sight-threatening."),
    2: ("Moderate NPDR detected with {conf:.0%} confidence. "
        "Grad-CAM focuses on the {zones}, identifying {kw}. "
        "Progressive microvascular damage — closer monitoring required."),
    3: ("Severe NPDR detected with {conf:.0%} confidence. "
        "Extensive activation across the {zones} corresponds to {kw}. "
        "High risk (>50%) of progression to PDR within 1 year."),
    4: ("PDR detected with {conf:.0%} confidence — most sight-threatening stage. "
        "Critical activation in the {zones} driven by {kw}. "
        "Immediate ophthalmological intervention essential."),
}

ZONE_DESCRIPTIONS = {
    "macula":     "macular region (central vision area)",
    "optic_disc": "optic disc region",
    "peripheral": "peripheral retina",
    "superior":   "superior retinal quadrant",
    "inferior":   "inferior retinal quadrant",
}

RISK_INFO = {
    0: {"level": "No risk",       "dot": "#639922"},
    1: {"level": "Low risk",      "dot": "#BA7517"},
    2: {"level": "Moderate risk", "dot": "#EF9F27"},
    3: {"level": "High risk",     "dot": "#E24B4A"},
    4: {"level": "Severe risk",   "dot": "#A32D2D"},
}

CLINICAL_ADVICE = {
    0: "Routine annual screening. Maintain HbA1c < 7%.",
    1: "Repeat screening in 9-12 months. Optimize glycemic and BP control.",
    2: "Ophthalmology referral within 3 months. Monitor for macular edema.",
    3: "Urgent referral within 1 month. Consider pan-retinal photocoagulation.",
    4: "IMMEDIATE referral. Laser / anti-VEGF / vitrectomy to prevent blindness.",
}

print("\n[Config] All settings loaded.")



[Config] All settings loaded.


In [11]:
import os
import sys
import zipfile
import pandas as pd

# Add these (missing in your previous config)
TRAIN_ZIP = os.path.join(DATA_ROOT, "train.zip")
TEST_ZIP  = os.path.join(DATA_ROOT, "test.zip")

# Define class names (modify if needed)
CLASS_NAMES = ["Class_0", "Class_1", "Class_2", "Class_3", "Class_4"]


def extract_zip(zip_path, extract_to):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f"[Data] '{extract_to}' already exists — skipping.")
        return

    if not os.path.exists(zip_path):
        print(f"[Warning] {zip_path} not found — skipping extraction.")
        return

    print(f"[Data] Extracting {zip_path} …")
    os.makedirs(extract_to, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)

    # Flatten single sub-folder if present
    entries = os.listdir(extract_to)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_to, entries[0])):
        sub = os.path.join(extract_to, entries[0])
        for f in os.listdir(sub):
            os.rename(os.path.join(sub, f), os.path.join(extract_to, f))
        os.rmdir(sub)

    n = len([
        f for f in os.listdir(extract_to)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])
    print(f"[Data] Extracted {n} images.")


def _find_image(img_dir, img_id):
    for ext in [".png", ".jpg", ".jpeg"]:
        p = os.path.join(img_dir, img_id + ext)
        if os.path.exists(p):
            return p
    return None


def load_data():
    if not os.path.exists(TRAIN_CSV):
        sys.exit(f"[ERROR] train.csv not found at '{TRAIN_CSV}'.")

    # Extract datasets
    extract_zip(TRAIN_ZIP, TRAIN_IMGS)
    extract_zip(TEST_ZIP, TEST_IMGS)

    # Load CSVs
    train_df = pd.read_csv(TRAIN_CSV)
    test_df  = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None

    print(f"\n[Data] Rows in train.csv : {len(train_df)}")
    print(f"[Data] Columns           : {train_df.columns.tolist()}")

    # Class distribution
    if "diagnosis" in train_df.columns:
        dist = train_df["diagnosis"].value_counts().sort_index()
        if len(CLASS_NAMES) == len(dist):
            dist.index = [CLASS_NAMES[i] for i in dist.index]
        print(f"[Data] Class distribution:\n{dist.to_string()}")
    else:
        print("[Warning] 'diagnosis' column not found.")

    return train_df, test_df


print("\n[Step 1/8] Loading dataset …")
train_df, test_df = load_data()


[Step 1/8] Loading dataset …
[Data] '.\train_images' already exists — skipping.
[Data] '.\test_images' already exists — skipping.

[Data] Rows in train.csv : 3662
[Data] Columns           : ['id_code', 'diagnosis']
[Data] Class distribution:
Class_0    1805
Class_1     370
Class_2     999
Class_3     193
Class_4     295


In [12]:
def _save_plot(fig, filename):
    path = os.path.join(PLOTS_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[Plot] → {path}")


def _find_img(img_dir, img_id):
    for ext in [".png", ".jpg", ".jpeg"]:
        p = os.path.join(img_dir, img_id + ext)
        if os.path.exists(p):
            return p
    return None


def _crop_black(img, tol=7):
    arr  = np.array(img)
    mask = arr.max(axis=2) > tol
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return img
    return Image.fromarray(arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1])


def plot_class_distribution(df):
    counts = df["diagnosis"].value_counts().sort_index()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("APTOS 2019 — Class Distribution", fontsize=14, fontweight="bold")
    bars = axes[0].bar(CLASS_SHORT, counts.values, color=BAR_COLORS, edgecolor="white")
    axes[0].set_xlabel("DR Class"); axes[0].set_ylabel("Image Count")
    axes[0].set_title("Images per class")
    for bar, c in zip(bars, counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                     str(c), ha="center", fontsize=10, fontweight="bold")
    axes[0].set_ylim(0, counts.max() * 1.15)
    axes[1].pie(counts.values, labels=CLASS_SHORT, colors=BAR_COLORS,
                autopct="%1.1f%%", startangle=140, pctdistance=0.78)
    axes[1].set_title("Class proportion")
    fig.tight_layout()
    _save_plot(fig, "01_class_distribution.png")


def plot_sample_images(df, img_dir, n_per_class=3):
    fig, axes = plt.subplots(5, n_per_class, figsize=(n_per_class * 3.5, 17))
    fig.suptitle("Sample Fundus Images per DR Class",
                 fontsize=14, fontweight="bold", y=1.01)
    for cls in range(5):
        sub = df[df["diagnosis"] == cls].sample(
            min(n_per_class, (df["diagnosis"] == cls).sum()), random_state=42)
        for j, (_, row) in enumerate(sub.iterrows()):
            ax   = axes[cls][j]
            path = _find_img(img_dir, str(row["id_code"]))
            if path:
                ax.imshow(Image.open(path).convert("RGB"))
            else:
                ax.set_facecolor("#111")
                ax.text(0.5, 0.5, "Not found", ha="center", va="center",
                        color="white", transform=ax.transAxes)
            ax.axis("off")
            if j == 0:
                ax.set_title(f"Class {cls}: {CLASS_NAMES[cls]}",
                             fontsize=11, fontweight="bold",
                             color=BAR_COLORS[cls], pad=6)
    fig.tight_layout()
    _save_plot(fig, "02_sample_images.png")


def plot_preprocessing(df, img_dir):
    sample = df.sample(3, random_state=7)
    fig, axes = plt.subplots(3, 3, figsize=(12, 11))
    fig.suptitle("Preprocessing: Raw → Black-border Crop → Resize 224×224",
                 fontsize=13, fontweight="bold")
    for ax, title in zip(axes[0], ["1. Raw", "2. Cropped", "3. Resized 224×224"]):
        ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    for i, (_, row) in enumerate(sample.iterrows()):
        p = _find_img(img_dir, str(row["id_code"]))
        if not p:
            [ax.axis("off") for ax in axes[i]]; continue
        raw     = Image.open(p).convert("RGB")
        cropped = _crop_black(raw)
        resized = cropped.resize((224, 224))
        for ax, img_show in zip(axes[i], [raw, cropped, resized]):
            ax.imshow(img_show); ax.axis("off")
            ax.set_xlabel(f"Shape: {np.array(img_show).shape}", fontsize=8)
    fig.tight_layout()
    _save_plot(fig, "03_preprocessing.png")


def plot_pixel_distribution(df, img_dir, n=50):
    r, g, b = [], [], []
    for _, row in df.sample(min(n, len(df)), random_state=0).iterrows():
        p = _find_img(img_dir, str(row["id_code"]))
        if p:
            arr = np.array(Image.open(p).convert("RGB").resize((224, 224)))
            r.extend(arr[:,:,0].flatten()); g.extend(arr[:,:,1].flatten())
            b.extend(arr[:,:,2].flatten())
    if not r:
        return
    bins = np.linspace(0, 255, 64)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(r, bins=bins, alpha=0.55, color="#C04040", label="Red",   density=True)
    ax.hist(g, bins=bins, alpha=0.55, color="#40A040", label="Green", density=True)
    ax.hist(b, bins=bins, alpha=0.55, color="#4060C0", label="Blue",  density=True)
    ax.set_xlabel("Pixel value"); ax.set_ylabel("Density")
    ax.set_title(f"Pixel Intensity Distribution ({n} sampled images)")
    ax.legend(); fig.tight_layout()
    _save_plot(fig, "04_pixel_distribution.png")


print("\n[Step 2] EDA plots...")
plot_class_distribution(train_df)
if os.path.exists(TRAIN_IMGS) and len(os.listdir(TRAIN_IMGS)) > 0:
    plot_sample_images(train_df, TRAIN_IMGS)
    plot_preprocessing(train_df, TRAIN_IMGS)
    plot_pixel_distribution(train_df, TRAIN_IMGS)
else:
    print("[EDA] No images found — skipping image plots.")


[Step 2] EDA plots...
[Plot] → ./plots\01_class_distribution.png
[Plot] → ./plots\02_sample_images.png
[Plot] → ./plots\03_preprocessing.png
[Plot] → ./plots\04_pixel_distribution.png


In [13]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

def preprocess_image(img_path, label, augment=False):
    img = tf.io.read_file(img_path)

    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])

    img = tf.image.resize(img, [CONFIG["img_size"], CONFIG["img_size"]])

    img = tf.cast(img, tf.float32) / 255.0

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)

    return img, label

def build_tf_dataset(df, img_dir, augment=False):
    paths, labels = [], []

    for _, row in df.iterrows():
        p = _find_image(img_dir, str(row["id_code"]))  # ✅ fixed name
        if p:
            paths.append(p)
            labels.append(int(row["diagnosis"]))

    paths_t  = tf.constant(paths)
    labels_t = tf.constant(labels)

    ds = tf.data.Dataset.from_tensor_slices((paths_t, labels_t))

    if augment:
        ds = ds.shuffle(buffer_size=len(paths))

    ds = ds.map(
        lambda p, l: preprocess_image(p, l, augment),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(CONFIG["batch_size"])
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds, labels

def build_datasets(train_df):
    tr_df, vl_df = train_test_split(
        train_df,
        test_size=CONFIG["val_split"],
        stratify=train_df["diagnosis"],
        random_state=CONFIG["seed"]
    )

    print(f"[Data] Train: {len(tr_df)}  |  Val: {len(vl_df)}")

    tr_ds, tr_labels = build_tf_dataset(tr_df, TRAIN_IMGS, augment=True)
    vl_ds, vl_labels = build_tf_dataset(vl_df, TRAIN_IMGS, augment=False)

    return tr_ds, vl_ds, tr_df, vl_df, tr_labels, vl_labels

print("\n[Step 3] Building tf.data pipelines (Preprocessing)...")

train_ds, val_ds, train_split, val_split, tr_labels, vl_labels = build_datasets(train_df)


[Step 3] Building tf.data pipelines (Preprocessing)...
[Data] Train: 2929  |  Val: 733


In [14]:
def build_efficientnet(num_classes=5):
    """EfficientNetB3 — best for small medical datasets."""
    base = EfficientNetB3(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True  # fine-tune all layers
    # Freeze first 80% of layers, train last 20%
    for layer in base.layers[:int(len(base.layers) * 0.8)]:
        layer.trainable = False

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(512, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inp, out, name="EfficientNetB3_DR")
    return model


def build_densenet(num_classes=5):
    """DenseNet121 — dense connections, good for feature reuse."""
    base = DenseNet121(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True
    for layer in base.layers[:int(len(base.layers) * 0.8)]:
        layer.trainable = False

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(512, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inp, out, name="DenseNet121_DR")
    return model


def build_resnet(num_classes=5):
    """ResNet50 — residual connections, strong baseline."""
    base = ResNet50(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True
    for layer in base.layers[:int(len(base.layers) * 0.8)]:
        layer.trainable = False

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(512, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inp, out, name="ResNet50_DR")
    return model

In [15]:
IMG_SIZE = CONFIG["img_size"]
EPOCHS   = CONFIG["epochs"]
LR       = CONFIG["lr"]
NUM_CLASSES = CONFIG["num_classes"]

In [16]:
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, ResNet50
from tensorflow.keras import layers, Model

# -------- EfficientNet --------
def build_efficientnet(num_classes):
    base = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(CONFIG["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs=base.input, outputs=outputs)


# -------- DenseNet --------
def build_densenet(num_classes):
    base = DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(CONFIG["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs=base.input, outputs=outputs)


# -------- ResNet --------
def build_resnet(num_classes):
    base = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = True

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(CONFIG["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs=base.input, outputs=outputs)

In [17]:
def get_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.arange(NUM_CLASSES)
    cw = compute_class_weight("balanced", classes=classes, y=labels)
    return dict(enumerate(cw))

In [18]:
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

def train_model(model, model_name, train_ds, val_ds, class_weights):
    print("\n" + "─"*60)
    print(f"Training : {model_name}")
    print("─"*60)

    save_path = os.path.join(MODELS_DIR, f"{model_name}_best.h5")
    history_csv = os.path.join(MODELS_DIR, f"{model_name}_history.csv")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        ModelCheckpoint(save_path, monitor="val_accuracy",
                        save_best_only=True, mode="max", verbose=1),

        EarlyStopping(monitor="val_accuracy", patience=5,
                      restore_best_weights=True, verbose=1),

        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),

        CSVLogger(history_csv)
    ]

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    print(f"\n[{model_name}] Done. Saved → {save_path}")
    return hist, save_path

In [19]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

def evaluate_model(model, model_name, val_ds, vl_labels):
    print(f"\n[{model_name}] Evaluating...")

    probs = model.predict(val_ds, verbose=0)
    preds = np.argmax(probs, axis=1)

    acc = accuracy_score(vl_labels[:len(preds)], preds)
    kappa = cohen_kappa_score(vl_labels[:len(preds)], preds, weights="quadratic")

    print(f"Accuracy : {acc:.4f}")
    print(f"Kappa    : {kappa:.4f}")
    print("\nClassification Report:")
    print(classification_report(vl_labels[:len(preds)], preds))

    return {"accuracy": acc, "kappa": kappa, "preds": preds}

In [ ]:
# Class weights
class_weights = get_class_weights(tr_labels)
print("[Weights]:", class_weights)


# -------- EfficientNet --------
print("\n[Step 4a] EfficientNet")
eff_model = build_efficientnet(NUM_CLASSES)
eff_hist, _ = train_model(eff_model, "EfficientNetB3", train_ds, val_ds, class_weights)
eff_result = evaluate_model(eff_model, "EfficientNetB3", val_ds, vl_labels)


# -------- DenseNet --------
print("\n[Step 4b] DenseNet")
dns_model = build_densenet(NUM_CLASSES)
dns_hist, _ = train_model(dns_model, "DenseNet121", train_ds, val_ds, class_weights)
dns_result = evaluate_model(dns_model, "DenseNet121", val_ds, vl_labels)


# -------- ResNet --------
print("\n[Step 4c] ResNet")
res_model = build_resnet(NUM_CLASSES)
res_hist, _ = train_model(res_model, "ResNet50", train_ds, val_ds, class_weights)
res_result = evaluate_model(res_model, "ResNet50", val_ds, vl_labels)

[Weights]: {0: np.float64(0.4056786703601108), 1: np.float64(1.979054054054054), 2: np.float64(0.7331664580725907), 3: np.float64(3.803896103896104), 4: np.float64(2.4822033898305085)}

[Step 4a] EfficientNet
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step

────────────────────────────────────────────────────────────
Training : EfficientNetB3
────────────────────────────────────────────────────────────
Epoch 1/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4202 - loss: 1.8883
Epoch 1: val_accuracy improved from None to 0.22647, saving model to ./saved_models\EfficientNetB3_best.h5



Epoch 1: finished saving model to ./saved_models\EfficientNetB3_best.h5
184/184 ━━━━━━━━━━━━━━━━━━━━ 880s 4s/step - accuracy: 0.5005 - loss: 1.6065 - val_accuracy: 0.2265 - val_loss: 1.7325 - learning_rate: 1.0000e-04
Epoch 2/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6241 - loss: 1.1677
Epoch 2: val_accuracy improved from 0.22647 to 0.40655, saving model to ./saved_models\EfficientNetB3_best.h5



Epoch 2: finished saving model to ./saved_models\EfficientNetB3_best.h5
184/184 ━━━━━━━━━━━━━━━━━━━━ 737s 4s/step - accuracy: 0.6282 - loss: 1.1977 - val_accuracy: 0.4065 - val_loss: 1.4890 - learning_rate: 1.0000e-04
Epoch 3/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6822 - loss: 0.9559
Epoch 3: val_accuracy did not improve from 0.40655
184/184 ━━━━━━━━━━━━━━━━━━━━ 777s 4s/step - accuracy: 0.6821 - loss: 0.9896 - val_accuracy: 0.1528 - val_loss: 1.7182 - learning_rate: 1.0000e-04
Epoch 4/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7091 - loss: 0.9105
Epoch 4: val_accuracy did not improve from 0.40655
184/184 ━━━━━━━━━━━━━━━━━━━━ 843s 5s/step - accuracy: 0.7177 - loss: 0.9047 - val_accuracy: 0.3329 - val_loss: 1.7686 - learning_rate: 1.0000e-04
Epoch 5/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7360 - loss: 0.8130
Epoch 5: val_accuracy improved from 0.40655 to 0.44065, saving model to ./saved_models\EfficientNetB3_best.h5



Epoch 5: finished saving model to ./saved_models\EfficientNetB3_best.h5
184/184 ━━━━━━━━━━━━━━━━━━━━ 832s 5s/step - accuracy: 0.7398 - loss: 0.8063 - val_accuracy: 0.4407 - val_loss: 1.4042 - learning_rate: 1.0000e-04
Epoch 6/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7543 - loss: 0.7768
Epoch 6: val_accuracy improved from 0.44065 to 0.44611, saving model to ./saved_models\EfficientNetB3_best.h5



Epoch 6: finished saving model to ./saved_models\EfficientNetB3_best.h5
184/184 ━━━━━━━━━━━━━━━━━━━━ 863s 5s/step - accuracy: 0.7501 - loss: 0.7949 - val_accuracy: 0.4461 - val_loss: 1399.1351 - learning_rate: 1.0000e-04
Epoch 7/20
 23/184 ━━━━━━━━━━━━━━━━━━━━ 9:48 4s/step - accuracy: 0.7516 - loss: 0.7147